# Notebook 09 — Model Selection: Four Candidate Models

## Project: AI-Supply-Chain-Digital-Marketing

This notebook performs the **final model-selection comparison** for the supply-chain disruption classifier.

Models compared:

1. Logistic Regression
2. Random Forest
3. XGBoost
4. LightGBM

### Selection protocol

- Training data: `X_train_preprocessed.csv` / `y_train_preprocessed.csv`
- Validation data: `X_validation_preprocessed.csv` / `y_validation_preprocessed.csv`
- Test data is loaded only for a **protection/integrity check** and is never used for fitting or selection.
- Primary selection metric: **Validation F1**
- Secondary metrics: Accuracy, Precision, Recall, ROC-AUC, PR-AUC
- The selected model is saved as a **selection decision only**. Final candidate training is performed in Notebook 10.
- No test predictions or test metrics are generated here.


In [1]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

try:
    from xgboost import XGBClassifier
except ImportError as e:
    raise ImportError(
        "XGBoost is required. Install with: %pip install xgboost"
    ) from e

try:
    from lightgbm import LGBMClassifier
except ImportError as e:
    raise ImportError(
        "LightGBM is required. Install with: %pip install lightgbm"
    ) from e


# Project directories
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = BASE_DIR / "data" / "processed"
MODEL_DIR = BASE_DIR / "models" / "supply_chain"
RESULTS_DIR = BASE_DIR / "results" / "metrics"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR.resolve())
print("Processed directory:", PROCESSED_DIR.exists())
print("Model directory:", MODEL_DIR.exists())
print("Results directory:", RESULTS_DIR.exists())


Base directory: C:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
Processed directory: True
Model directory: True
Results directory: True


## Cell 3 — Load the leakage-safe train/validation/test splits

In [2]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_preprocessed.csv")
y_train = pd.read_csv(PROCESSED_DIR / "y_train_preprocessed.csv").squeeze("columns")

X_validation = pd.read_csv(PROCESSED_DIR / "X_validation_preprocessed.csv")
y_validation = pd.read_csv(PROCESSED_DIR / "y_validation_preprocessed.csv").squeeze("columns")

# Test is loaded only to verify that it remains untouched.
X_test = pd.read_csv(PROCESSED_DIR / "X_test_preprocessed.csv")
y_test = pd.read_csv(PROCESSED_DIR / "y_test_preprocessed.csv").squeeze("columns")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


X_train: (3500, 590)
y_train: (3500,)
X_validation: (750, 590)
y_validation: (750,)
X_test: (750, 590)
y_test: (750,)


## Cell 4 — Validate split integrity before model comparison

In [3]:
assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert len(y_train) == 3500
assert len(y_validation) == 750
assert len(y_test) == 750

assert list(X_train.columns) == list(X_validation.columns) == list(X_test.columns)

assert not X_train.isna().any().any()
assert not X_validation.isna().any().any()
assert not X_test.isna().any().any()

assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_validation.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

# Verify binary target
assert set(pd.Series(y_train).unique()).issubset({0, 1})
assert set(pd.Series(y_validation).unique()).issubset({0, 1})
assert set(pd.Series(y_test).unique()).issubset({0, 1})

print("Split integrity: PASS")
print("Predictor columns:", X_train.shape[1])
print("Training target distribution:")
print(pd.Series(y_train).value_counts().sort_index())


Split integrity: PASS
Predictor columns: 590
Training target distribution:
Disruption_Occurred
0    1343
1    2157
Name: count, dtype: int64


## Cell 5 — Define the four candidate models

These are baseline candidate configurations for **model selection**, not final hyperparameter tuning. Hyperparameter tuning is reserved for Notebook 11.


In [4]:
model_configs = {
    "Logistic Regression": lambda: LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": lambda: RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),

    "XGBoost": lambda: XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM": lambda: LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
}

expected_models = {
    "Logistic Regression",
    "Random Forest",
    "XGBoost",
    "LightGBM"
}

assert set(model_configs) == expected_models

print("Four candidate models defined: PASS")
print("\n".join(f"- {name}" for name in model_configs))


Four candidate models defined: PASS
- Logistic Regression
- Random Forest
- XGBoost
- LightGBM


## Cell 6 — Train each candidate on training data and evaluate on validation data

**Important:** only the training split is used in `.fit()`. Validation data is used only for model comparison.

The test split is not passed to any model.


In [5]:
comparison_rows = []
validation_predictions = {}
validation_probabilities = {}
trained_candidates = {}

for model_name, model_factory in model_configs.items():
    print(f"Training {model_name}...")

    model = model_factory()

    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    training_seconds = time.perf_counter() - start_time

    y_pred = model.predict(X_validation)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_validation)[:, 1]
    elif hasattr(model, "decision_function"):
        scores = model.decision_function(X_validation)
        y_prob = (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    else:
        raise AttributeError(
            f"{model_name} does not provide predict_proba or decision_function."
        )

    validation_predictions[model_name] = y_pred
    validation_probabilities[model_name] = y_prob
    trained_candidates[model_name] = model

    row = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_validation, y_pred),
        "Precision": precision_score(y_validation, y_pred, zero_division=0),
        "Recall": recall_score(y_validation, y_pred, zero_division=0),
        "F1": f1_score(y_validation, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_validation, y_prob),
        "PR_AUC": average_precision_score(y_validation, y_prob),
        "Training_Time_Seconds": training_seconds
    }

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

print("\nFour-model validation comparison:")
display(comparison_df.round(6))


Training Logistic Regression...
Training Random Forest...
Training XGBoost...
Training LightGBM...

Four-model validation comparison:


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time_Seconds
0,Logistic Regression,0.756000,0.872928,0.697572,0.775460,0.845081,0.907606,0.287422
1,Random Forest,0.748000,0.812796,0.757174,0.784000,0.830234,0.897670,0.766011
2,XGBoost,0.766667,0.799569,0.818985,0.809160,0.839937,0.904508,2.689540
3,LightGBM,0.757333,0.799117,0.799117,0.799117,0.833753,0.900834,0.290114


## Cell 7 — Rank models using validation F1

F1 is the project's primary selection metric because the disruption target contains both disruption and non-disruption cases, and the project needs a balance between precision and recall.

No test metric is used for this decision.


In [6]:
comparison_df = comparison_df.sort_values(
    by=["F1", "ROC_AUC", "PR_AUC"],
    ascending=[False, False, False]
).reset_index(drop=True)

comparison_df.insert(0, "Rank", np.arange(1, len(comparison_df) + 1))

selected_model_name = comparison_df.loc[0, "Model"]

assert len(comparison_df) == 4
assert set(comparison_df["Model"]) == expected_models
assert selected_model_name in expected_models

print("Model ranking by validation F1:")
display(comparison_df.round(6))

print("Selected model:", selected_model_name)
print("Primary selection metric: Validation F1")
print("Selected validation F1:", round(float(comparison_df.loc[0, "F1"]), 6))


Model ranking by validation F1:


,Rank,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,Training_Time_Seconds
0,1,XGBoost,0.766667,0.799569,0.818985,0.809160,0.839937,0.904508,2.689540
1,2,LightGBM,0.757333,0.799117,0.799117,0.799117,0.833753,0.900834,0.290114
2,3,Random Forest,0.748000,0.812796,0.757174,0.784000,0.830234,0.897670,0.766011
3,4,Logistic Regression,0.756000,0.872928,0.697572,0.775460,0.845081,0.907606,0.287422


Selected model: XGBoost
Primary selection metric: Validation F1
Selected validation F1: 0.80916


## Cell 8 — Check that the selected model is actually the highest-F1 candidate

This prevents an accidental selection from an older two-model comparison artifact.


In [7]:
best_f1 = comparison_df["F1"].max()
selected_f1 = float(comparison_df.loc[comparison_df["Model"] == selected_model_name, "F1"].iloc[0])

assert np.isclose(selected_f1, best_f1)

# Explicitly verify that the selection came from all four candidates.
assert comparison_df["Model"].nunique() == 4

print("Four-model selection verification: PASS")
print("Selected model:", selected_model_name)
print("Best validation F1:", round(float(best_f1), 6))


Four-model selection verification: PASS
Selected model: XGBoost
Best validation F1: 0.80916


## Cell 9 — Save the final four-model selection artifact

This file is the **only model-selection artifact Notebook 10 should use**.

It contains the actual validation comparison generated by this notebook. No manually entered performance values are used.


In [8]:
selection_path = MODEL_DIR / "all_four_model_selection_info.json"

ranking_records = []
for _, row in comparison_df.iterrows():
    ranking_records.append({
        "rank": int(row["Rank"]),
        "model": row["Model"],
        "f1": float(row["F1"]),
        "accuracy": float(row["Accuracy"]),
        "precision": float(row["Precision"]),
        "recall": float(row["Recall"]),
        "roc_auc": float(row["ROC_AUC"]),
        "pr_auc": float(row["PR_AUC"]),
        "training_time_seconds": float(row["Training_Time_Seconds"])
    })

selection_info = {
    "selection_metric": "F1",
    "selection_split": "validation",
    "selected_model": selected_model_name,
    "ranking": ranking_records,
    "models_compared": sorted(expected_models),
    "test_used_for_training": False,
    "test_used_for_selection": False,
    "test_predictions_generated": False
}

with open(selection_path, "w", encoding="utf-8") as f:
    json.dump(selection_info, f, indent=2)

print("Selection artifact saved:", selection_path)
print("Exists:", selection_path.exists())


Selection artifact saved: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\all_four_model_selection_info.json
Exists: True


## Cell 10 — Save the validation comparison table

This is an evidence table for the experiment. It contains results generated by the current run; it does not contain prefilled/fabricated values.


In [9]:
comparison_csv_path = RESULTS_DIR / "model_selection_four_models_validation.csv"

comparison_df.to_csv(comparison_csv_path, index=False)

print("Comparison table saved:", comparison_csv_path)
print("Exists:", comparison_csv_path.exists())


Comparison table saved: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\results\metrics\model_selection_four_models_validation.csv
Exists: True


## Cell 11 — Test-set protection verification

The test set must remain completely untouched until the final evaluation notebook.


In [10]:
test_prediction_generated = False
test_used_for_training = False
test_used_for_selection = False

assert test_prediction_generated is False
assert test_used_for_training is False
assert test_used_for_selection is False

# Confirm the selection table contains validation metrics only.
required_columns = {
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC_AUC",
    "PR_AUC",
    "Training_Time_Seconds"
}
assert required_columns.issubset(comparison_df.columns)
assert not any("Test" in column for column in comparison_df.columns)

print("Test-set protection: PASS")
print("Test prediction generated: NO")
print("Test used for training: NO")
print("Test used for model selection: NO")


Test-set protection: PASS
Test prediction generated: NO
Test used for training: NO
Test used for model selection: NO


## Cell 12 — Reload the selection artifact and verify it

In [11]:
with open(selection_path, "r", encoding="utf-8") as f:
    saved_selection = json.load(f)

assert saved_selection["selected_model"] == selected_model_name
assert saved_selection["selection_metric"] == "F1"
assert saved_selection["selection_split"] == "validation"
assert set(saved_selection["models_compared"]) == expected_models
assert saved_selection["test_used_for_training"] is False
assert saved_selection["test_used_for_selection"] is False
assert saved_selection["test_predictions_generated"] is False
assert len(saved_selection["ranking"]) == 4

print("Saved selection artifact verification: PASS")
print("Selected model:", saved_selection["selected_model"])
print("Selection metric:", saved_selection["selection_metric"])
print("Models compared:", saved_selection["models_compared"])


Saved selection artifact verification: PASS
Selected model: XGBoost
Selection metric: F1
Models compared: ['LightGBM', 'Logistic Regression', 'Random Forest', 'XGBoost']


## Cell 13 — Important hand-off to Notebook 10

Notebook 10 should now read:

`models/supply_chain/all_four_model_selection_info.json`

It must **not** read:

- `model_selection_info.json`
- `xgb_lightgbm_selection_info.json`

Those are older/partial selection artifacts and can cause an inconsistent model choice.


In [12]:
print("Notebook 10 hand-off file:")
print(selection_path)

print("\nSelected candidate:", selected_model_name)
print("Primary metric:", selection_info["selection_metric"])
print("Selection split:", selection_info["selection_split"])
print("Test used for selection:", selection_info["test_used_for_selection"])


Notebook 10 hand-off file:
c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\models\supply_chain\all_four_model_selection_info.json

Selected candidate: XGBoost
Primary metric: F1
Selection split: validation
Test used for selection: False


## Cell 14 — Final validation

In [13]:
assert set(model_configs) == expected_models
assert len(comparison_df) == 4
assert comparison_df["Model"].nunique() == 4

assert selected_model_name in expected_models
assert selected_model_name == selection_info["selected_model"]

assert selection_info["selection_metric"] == "F1"
assert selection_info["selection_split"] == "validation"

assert selection_info["test_used_for_training"] is False
assert selection_info["test_used_for_selection"] is False
assert selection_info["test_predictions_generated"] is False

assert selection_path.exists()
assert comparison_csv_path.exists()

assert X_train.shape == (3500, 590)
assert X_validation.shape == (750, 590)
assert X_test.shape == (750, 590)

assert not any("Test" in column for column in comparison_df.columns)

print("Models compared:", len(comparison_df))
print("Selected model:", selected_model_name)
print("Selection metric:", selection_info["selection_metric"])
print("Training rows:", X_train.shape[0])
print("Validation rows:", X_validation.shape[0])
print("Test rows:", X_test.shape[0])
print("Test used for training: False")
print("Test used for selection: False")
print("Test predictions generated: False")
print("Selection artifact exists:", selection_path.exists())
print("Comparison table exists:", comparison_csv_path.exists())
print("\nNotebook 09 final validation: PASS")


Models compared: 4
Selected model: XGBoost
Selection metric: F1
Training rows: 3500
Validation rows: 750
Test rows: 750
Test used for training: False
Test used for selection: False
Test predictions generated: False
Selection artifact exists: True
Comparison table exists: True

Notebook 09 final validation: PASS
